# Silver → Gold 

## 1. Configuração

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## 2. Dimensão Motivo
### Regra de negócio e estrutura
A dimensão de motivos disponibiliza os atributos usados para classificar os motivos de atendimento:
- id_motivo: identificador do motivo e chave da dimensão;
- nome_motivo: descrição do motivo;
- categoria: categoria padronizada na Silver;
- criticidade: nível de criticidade padronizado na Silver.
A etapa Silver já converte o identificador para inteiro, remove espaços extras, trata categorias ausentes conhecidas e padroniza acentuação e criticidade.

In [0]:
df_silver_motivos = spark.table(f"{silver_schema}.tb_motivos")

df_gold_motivo = (
    df_silver_motivos
    .select(
        F.col("id_motivo"),
        F.col("nome_motivo"),
        F.col("categoria"),
        F.col("criticidade")
    )
)
(
    df_gold_motivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_motivo")
)

print(f"Tabela {gold_schema}.dim_motivo salva com sucesso!")
print(f"Total de motivos na dimensão: {df_gold_motivo.count()}")

display(df_gold_motivo.orderBy("id_motivo"))

## 3. Dimensão Canal
###Regra de negócio e estrutura
A dimensão de canais disponibiliza os atributos de identificação e classificação dos canais:
- id_canal: identificador do canal e chave da dimensão;
- nome_canal: nome do canal;
- tipo_canal: tipo ou classificação do canal;
- status_canal: status cadastral do canal.
A dimensão utiliza a tabela de canais já tratada na Silver. A seleção dos atributos mantém o escopo definido para a dimensão e evita repetir nesta etapa as transformações de limpeza realizadas anteriormente.

In [0]:
df_silver_canais = spark.table(f"{silver_schema}.canais")

df_gold_canal = (
    df_silver_canais
    .select(
        F.col("id_canal"),
        F.col("nome_canal"),
        F.col("tipo_canal"),
        F.col("status_canal")
    )
)
(
    df_gold_canal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_canal")
)

print(f"Tabela {gold_schema}.dim_canal salva com sucesso!")
print(f"Total de canais na dimensão: {df_gold_canal.count()}")

display(df_gold_canal.orderBy("id_canal"))

## fact_chamados

In [0]:
df_pesquisa_satisfacao = (
    spark.table(f"{silver_schema}.pesquisa_satisfacao")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_custo = (
    spark.table(f"{silver_schema}.custos")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_chamados = (
    spark.table(f"{silver_schema}.tb_chamados")
    .withColumn("id_chamado", F.col("id_chamado").cast("string"))
    .dropDuplicates(["id_chamado"])
)

df_fact = (
    df_chamados
    .join(df_pesquisa_satisfacao, on="id_chamado", how="left")
    .join(df_custo, on="id_chamado", how="left")
    .join(df_gold_motivos.withColumnRenamed("nome_motivo","motivo"), on="motivo", how="left")
    .join(df_gold_canais.withColumnRenamed("nome_canais","canal"), on="canal", how="left")
    .select(
        "id_chamado",
        F.col("id_cliente").cast("decimal(18,2)").alias("id_cliente"),
        F.col("id_atendente").cast("decimal(18,2)").alias("id_atendente"),
        F.col("id_motivo").cast("decimal(18,2)").alias("id_motivo"),
        F.col("id_canal").cast("decimal(18,2)").alias("id_canal"),
        F.col("id_data").cast("decimal(18,2)").alias("id_data"),
        F.col("resolvido").cast("decimal(18,2)").alias("resolvido"),
        F.col("custo").cast("decimal(18,2)").alias("custo"),
        F.col("nota_atendimento").cast("decimal(18,2)").alias("nota_atendimento"),
        F.col("tempo_espera").cast("decimal(18,2)").alias("tempo_espera_seg"),
        F.col("tempo_atendimento").cast("decimal(18,2)").alias("tempo_atendimento_seg")
    )
)

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_chamados")

print(f"fact_chamados: {df_fact.count()} chamados")
display(df_fact.limit(10))